# Life tables compared

A `life_table` lifespan draws a person's age at death from a period life table: for each age, the chance of dying before the next birthday. This notebook puts every life table there is to choose from side by side, so they can be compared before one is named in `lifespan.csv`.

The tables come from three places: the code repository's `templates/aux/`, the data repo's `aux/`, and any scenario's own `aux/`. `life_table.csv` is the default (shown as `default`), and `life_table__<name>.csv` is one a `lifespan.csv` row names in its `life_table` column. A name found in more than one place is shown with its location after the first. Each file can hold several vintages; the latest is shown.

The same tables from the terminal: `analysis sim lifespan tables`, and for one age, `analysis sim lifespan check AGE`.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from analysis.sim import lifespan_tables
from analysis.sim.notebook_ui.life_table_widgets import at_age_widget
from analysis.sim.notebook_ui.pickers import find_data_dir

DATA_DIR = find_data_dir()

## The tables found

In [ ]:
TABLES = lifespan_tables.find_tables(DATA_DIR)
pd.DataFrame(
    [(t.label, t.location, lifespan_tables.read_table(t.path)[0]) for t in TABLES],
    columns=["Table", "Location", "Effective"],
).set_index("Table")

## Their values at each age

For each table and sex, two columns:

- **q**: the chance that someone who has just turned that age dies before the next birthday.
- **expected**: the expected age at death for someone who has just turned that age. It is read the way the simulator reads the table: the yearly `q` spread evenly over the year's months, a death falling mid-month on average, and nobody living past 130. It is always more than the age itself, and it rises with age, since reaching an age means having escaped the deaths before it.

A table that ends sooner than another is blank (`-`) past its last age; past its end, everyone dies within the month.

In [ ]:
COMPARISON = lifespan_tables.comparison(DATA_DIR)
COMPARISON.style.format(
    {c: "{:.6f}" if c[-1] == "q" else "{:.2f}" for c in COMPARISON.columns}, na_rep="-",
)

## At one age

Each table's values at the age in the box, which may be fractional (`67.5`): `q` and the expected age at death are interpolated between the whole ages either side. `years left` is the expected age at death less the age. The table is redrawn as the age changes.

In [ ]:
at_age_widget(DATA_DIR, age=65)